In [14]:
from pathlib import Path
import sqlite3
import pandas as pd
from prefect import flow, task
from prefect.logging import get_run_logger

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

PROJECT_ROOT = BASE_DIR.parent
DB_PATH = PROJECT_ROOT / "data" / "shopdata.db"

OUTPUT_DB_PATH = BASE_DIR / "analytics.db"
CLEAN_CUSTOMERS_CSV = BASE_DIR / "clean_customers.csv"
CLEAN_ORDERS_CSV = BASE_DIR / "clean_orders.csv"


@task(retries=2, retry_delay_seconds=5)
def extract(db_path: str) -> dict:
    logger = get_run_logger()
    logger.info(f"Starting extract from {db_path}")

    try:
        tables = ["vw_raw_customers", "vw_raw_orders", "vw_exchange_rates"]
        extracted_data = {}

        with sqlite3.connect(db_path) as conn:
            for table in tables:
                query = f"SELECT * FROM {table}"
                df = pd.read_sql_query(query, conn)
                extracted_data[table] = df
                logger.info(f"Loaded {len(df)} rows from {table}")

        logger.info("Extract completed successfully")
        return extracted_data

    except Exception:
        logger.exception("Extract failed")
        raise
        
@task
def transform_customers(customers_df: pd.DataFrame) -> pd.DataFrame:
    logger = get_run_logger()
    logger.info(f"Starting customer transform with {len(customers_df)} rows")

    try:
        customers_df = customers_df.copy()

        customers_df["signup_date"] = pd.to_datetime(
            customers_df["signup_date"], errors="coerce"
        ).dt.date

        before_dedup = len(customers_df)
        customers_df = customers_df.sort_values(
            by=["customer_id", "signup_date"],
            ascending=[True, False]
        )
        customers_df = customers_df.drop_duplicates(
            subset=["customer_id"],
            keep="first"
        )
        after_dedup = len(customers_df)
        logger.info(
            f"Customer dedup complete: removed {before_dedup - after_dedup} duplicate rows"
        )

        phone_mask = customers_df["phone"].notna()
        customers_df.loc[phone_mask, "phone"] = (
            customers_df.loc[phone_mask, "phone"]
            .astype(str)
            .str.replace(r"\D+", "", regex=True)
        )
        customers_df["phone"] = customers_df["phone"].replace("", pd.NA)

        missing_email_count = customers_df["email"].isna().sum()
        customers_df["email"] = customers_df["email"].fillna("unknown@domain.com")
        logger.info(f"Filled {missing_email_count} missing email values")

        logger.info(f"Customer transform completed with {len(customers_df)} rows")
        return customers_df

    except Exception:
        logger.exception("Customer transform failed")
        raise



@flow(name="etl-pipeline")
def etl_pipeline(db_path: str = str(DB_PATH)) -> None:
    logger = get_run_logger()
    logger.info("ETL pipeline started")

    try:
        data = extract(db_path)
        
        customers_df = transform_customers(data["vw_raw_customers"])

    except Exception:
        logger.exception("ETL pipeline failed")
        raise


if __name__ == "__main__":
    etl_pipeline()


09:12:55.488 | INFO    | Flow run 'crimson-crab' - Beginning flow run 'crimson-crab' for flow 'etl-pipeline'

09:12:55.501 | INFO    | Flow run 'crimson-crab' - ETL pipeline started

09:12:55.509 | INFO    | Task run 'extract-9f6' - Starting extract from /Users/earnch/projects/Data Engineer Technical Assignment/data/shopdata.db

09:12:55.536 | INFO    | Task run 'extract-9f6' - Loaded 12 rows from vw_raw_customers

09:12:55.540 | INFO    | Task run 'extract-9f6' - Loaded 20 rows from vw_raw_orders

...

09:12:55.572 | INFO    | Task run 'transform_customers-593' - Customer dedup complete: removed 2 duplicate rows

09:12:55.577 | INFO    | Task run 'transform_customers-593' - Filled 1 missing email values

09:12:55.578 | INFO    | Task run 'transform_customers-593' - Customer transform completed with 10 rows

09:12:55.583 | INFO    | Task run 'transform_customers-593' - Finished in state Completed()

09:12:56.503 | INFO    | Flow run 'crimson-crab' - Finished in state Completed()